Display model predictions dynamically by implementing custom caching mechanisms to improve performance, ensuring smooth updates without unnecessary reloading, and handling concurrent user requests efficiently.

In [20]:
import pandas as pd
import numpy as np
import joblib
import json
import time
import threading

from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor

In [21]:
df = pd.read_csv("heart_disease_train.csv")

model = joblib.load("saved_models/heart_disease_model_v1.joblib")

print("Dataset loaded:", df.shape)
print("Model loaded successfully!")

Dataset loaded: (79, 19)
Model loaded successfully!


In [22]:
target = "target"

if hasattr(model, "feature_names_in_"):
    features = list(model.feature_names_in_)

else:
    with open("saved_models/heart_disease_model_v1_metadata.json") as f:
        metadata = json.load(f)

    print("Metadata:", metadata)

    features = metadata.get("features")

    if features is None:
        features = metadata.get("feature_names")

    if features is None:
        raise ValueError("Model feature names not found in metadata.")

print("Expected model features:", features)
print("Number of features:", len(features))

Metadata: {'model_name': 'Logistic Regression', 'model_version': 'v1', 'created_at_utc': '2026-10-21T11:27:05.419958+00:00', 'features': ['age', 'trestbps', 'chol', 'thalach', 'oldpeak'], 'target': 'target', 'validation_metrics': {'precision': 0.9166666666666666, 'recall': 0.9166666666666666, 'f1_score': 0.9166666666666666, 'custom_score': 0.9166666666666667}, 'test_metrics': {'precision': 0.7692307692307693, 'recall': 0.8333333333333334, 'f1_score': 0.8}, 'library_versions': {'python': '3.13.14', 'scikit_learn': '1.9.0', 'numpy': '2.5.1', 'pandas': '3.0.3', 'joblib': '1.5.3'}, 'sha256': 'f494b3057e23240729586e27e436641729b003ab554098fef73e57efabcdc687'}
Expected model features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Number of features: 5


In [23]:
def validate_input(user_data):

    errors = {}
    cleaned_data = {}

    for feature in features:

        value = user_data.get(feature)

        if value is None or str(value).strip() == "":
            errors[feature] = "Missing value"
            continue

        try:
            if pd.api.types.is_numeric_dtype(df[feature]):
                value = float(value)

                if not np.isfinite(value):
                    errors[feature] = "Invalid numeric value"
                    continue

                minimum = df[feature].min()
                maximum = df[feature].max()

                if value < minimum or value > maximum:
                    errors[feature] = f"Enter value between {minimum} and {maximum}"
                    continue

            else:
                value = str(value)

                if value not in df[feature].astype(str).unique():
                    errors[feature] = "Invalid category"
                    continue

            cleaned_data[feature] = value

        except (ValueError, TypeError):
            errors[feature] = "Incorrect input format"

    if errors:
        return None, errors

    return pd.DataFrame([cleaned_data], columns=features), {}

In [24]:
prediction_cache = OrderedDict()
cache_lock = threading.Lock()

MAX_CACHE_SIZE = 100

def cached_prediction(input_data):

    key = tuple(input_data.iloc[0].tolist())

    with cache_lock:
        if key in prediction_cache:
            prediction_cache.move_to_end(key)
            return prediction_cache[key], "Cache"

    prediction = model.predict(input_data)[0]

    with cache_lock:
        prediction_cache[key] = prediction
        prediction_cache.move_to_end(key)

        if len(prediction_cache) > MAX_CACHE_SIZE:
            prediction_cache.popitem(last=False)

    return prediction, "Model"

In [26]:
user_data = {}

for feature in features:
    user_data[feature] = input(f"Enter {feature}: ")

validated_input, errors = validate_input(user_data)

if errors:
    print("Input Validation Failed:")
    print(errors)

else:
    print("All inputs are valid!")
    display(validated_input)

All inputs are valid!


,age,trestbps,chol,thalach,oldpeak
0,30.0,120.0,200.0,150.0,1.2


In [27]:
if validated_input is not None:

    prediction, source = cached_prediction(validated_input)

    print("Prediction:", prediction)
    print("Prediction Source:", source)

else:
    print("Prediction cancelled due to invalid inputs.")

c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Prediction: 0
Prediction Source: Model


In [28]:
if validated_input is not None:

    start = time.perf_counter()
    result1 = cached_prediction(validated_input)
    first_time = time.perf_counter() - start

    start = time.perf_counter()
    result2 = cached_prediction(validated_input)
    second_time = time.perf_counter() - start

    print("First Prediction:", result1)
    print("Second Prediction:", result2)

    print("First Time:", round(first_time * 1000, 3), "ms")
    print("Cached Time:", round(second_time * 1000, 3), "ms")

First Prediction: (np.int64(0), 'Cache')
Second Prediction: (np.int64(0), 'Cache')
First Time: 0.213 ms
Cached Time: 0.111 ms


In [29]:
def process_request(data):

    validated, errors = validate_input(data)

    if errors:
        return {"Status": "Invalid", "Errors": errors}

    prediction, source = cached_prediction(validated)

    return {
        "Status": "Success",
        "Prediction": str(prediction),
        "Source": source
    }


requests = [user_data, user_data, user_data]

with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(process_request, requests))

for i, result in enumerate(results, 1):
    print(f"Request {i}:", result)

Request 1: {'Status': 'Success', 'Prediction': '0', 'Source': 'Cache'}
Request 2: {'Status': 'Success', 'Prediction': '0', 'Source': 'Cache'}
Request 3: {'Status': 'Success', 'Prediction': '0', 'Source': 'Cache'}


In [30]:
print("Total Cached Predictions:", len(prediction_cache))
print("Maximum Cache Capacity:", MAX_CACHE_SIZE)

Total Cached Predictions: 1
Maximum Cache Capacity: 100
